# K513 · Week 3, Session 1
## Relationships between variables — putting a number on it

On Tuesday you drew two variables against each other and read the picture. Thursday you reshaped
tables so the pictures were drawable at all. Today you put a **number** on a relationship — and a
second number on how much to trust the first one.

Three questions, three tests:

| What you have | The test |
|---|---|
| two numbers | `pearsonr()` or `spearmanr()` |
| one number, one category | `f_oneway()` |
| two categories | `chi2_contingency()` |

All three come from `scipy.stats`, all three return **two** values, and the whole session is about
choosing between them — and about never letting the number replace the picture.

---
### Before you type anything

**File → Save a copy in Drive.**

This notebook is read-only for you. You can type into it and run it and it will look completely
normal, but nothing you do will be saved. Save your own copy first, every time.

---

### Using AI in this notebook

Gemini is built into Colab and you are welcome to use it here. Two things worth knowing:

- It does not know which columns you have or what we covered in class. Whatever it writes, you own.
- The most useful thing you can ask it is **"explain what this line does"** — not "write it for me".

There is a specific trap in this session. An AI will happily run a test for you and report a
p-value, and it has no way of knowing whether that test was the right one for your two columns —
that decision depends on the shape of your data, which it cannot see. Choosing the test is the part
that is yours.

---

### Turn off Unwanted AI Assistance

AI-powered coding completion is turned on by default. It is convenient but does not give you a chance
to think and learn. Turning it off helps you learn. You can always turn it back on when needed.
- Tools → settings → AI Assistance → Uncheck "Show AI-powered inline code completions"
- Tools → settings → Uncheck "Show context-powered code completions"

---

### How to run a cell

Click on a cell, then press **Shift + Enter**. That runs it and moves you to the next one. If
anything ever looks wrong: **Runtime → Restart session and run all**.


---
## 1 · Setup

Two imports you know, one that is new, and the two data addresses. Run these; neither cell shows
anything.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr, f_oneway, chi2_contingency

pd.set_option('display.precision', 3)

In [ ]:
BOSTON_URL = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/BostonHousing.csv"
BIKE_URL   = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/bikeshare.csv"

boston_df = pd.read_csv(BOSTON_URL)
bike_df   = pd.read_csv(BIKE_URL)

`scipy` is a scientific-computing library; `scipy.stats` is the part of it that holds statistical
tests. You import the individual tests by name, which is why the import line lists four of them.

There is a second library — `statsmodels` — that does fuller statistical modelling. It is heavier and
you do not need it here. Everything this term is one test at a time, which is exactly what
`scipy.stats` is for.


---
## 2 · Two numbers

### What r is

`r` is one number for how tightly two columns track each other. It is always between **−1 and +1**.
The **sign** is the direction, the **size** is the strength.

Start with the pair you met on Tuesday.


In [ ]:
r, p_value = pearsonr(bike_df['temp in Celsius'], bike_df['num_shared'])
print(f"Pearson r: {r:.3f},  p-value: {p_value:.3g}")

`pearsonr()` returns **two** values and you must unpack both. If you write `r = pearsonr(...)` you
get a result object rather than a number, and every later line that expects a number will fail.

Now two columns that barely relate at all.


In [ ]:
for col in ['humidity', 'windspeed', 'temp in Celsius']:
    r, p_value = pearsonr(bike_df[col], bike_df['num_shared'])
    print(f"{col:>16}   r = {r:+.3f}   p = {p_value:.3g}")

Draw all three, because a coefficient without a picture is a number you have decided to trust.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['humidity', 'windspeed', 'temp in Celsius']):
    sns.scatterplot(data=bike_df, x=col, y='num_shared', alpha=0.4, ax=ax)
plt.show()

`r = 0` means no **straight-line** relationship. That is not the same as no relationship, and it is
the single most expensive misreading of this number.

### Pearson or Spearman?

`pearsonr()` is computed on the **values**. `spearmanr()` throws the values away and uses their
**ranks** — 1st, 2nd, 3rd — so a value of 89 in a column whose next largest is 9 counts simply as
"the largest one".

That difference matters whenever a column has a long tail. Here is one that does.


In [ ]:
boston_df.skew(numeric_only=True).sort_values()

`CRIM` comes in at **5.22** and `MEDV` at **1.11**. The rule:

> **`skew()` between −1 and 1** → the column is close enough to symmetric → **`pearsonr()`**
> **outside −1 and 1** → the column is skewed → **`spearmanr()`**
> **either column fails** → the *pair* fails. One skewed column is enough to rule Pearson out.

Ordinal data — survey scales, ratings, education level — goes to `spearmanr()` whatever the skew.
Ranks are what it has.

Watch what the choice is worth on `CRIM` against `MEDV`.


In [ ]:
r_p, p_p = pearsonr(boston_df['CRIM'], boston_df['MEDV'])
r_s, p_s = spearmanr(boston_df['CRIM'], boston_df['MEDV'])

print(f"Pearson    r = {r_p:+.3f}   p = {p_p:.3g}")
print(f"Spearman rho = {r_s:+.3f}   p = {p_s:.3g}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(data=boston_df, x='CRIM', bins=40, ax=axes[0])
sns.scatterplot(data=boston_df, x='CRIM', y='MEDV', alpha=0.5, ax=axes[1])
plt.show()

Eleven tracts out of 506 carry crime rates above 25, and one is at 89. Those eleven drag Pearson
toward zero. Ranks cannot be dragged, so Spearman reports **−0.559**. Report that one here — not because it is
the bigger number, but because Pearson's −0.388 is describing those eleven tracts more than it is
describing the other 495.

**When the two numbers are far apart, go and look at the columns.** It is the cheapest diagnostic
in this session.

### The trap: neither one can see a curve

Back to `temp` and `num_shared`. Both columns are close to symmetric, so the rule says Pearson, and the
rule is right.


In [ ]:
r_p, _ = pearsonr(bike_df['temp in Celsius'], bike_df['num_shared'])
r_s, _ = spearmanr(bike_df['temp in Celsius'], bike_df['num_shared'])
print(f"Pearson  {r_p:+.3f}")
print(f"Spearman {r_s:+.3f}")

bands = pd.cut(bike_df['temp in Celsius'],
               bins=[-99, 10, 15, 20, 25, 30, 99],
               labels=['< 10', '10-15', '15-20', '20-25', '25-30', '> 30'])
bike_df.groupby(bands, observed=True)['num_shared'].mean().round(0)

The two coefficients agree almost perfectly — and **both of them miss the fall-off after 27°**.
Mean rides climb to 5,726 in the 25–30 band and then drop to 5,410 above 30.

The skew rule picks the **coefficient**. It cannot pick up the shape of the **relationship**. Only
the picture does that, which is why the scatterplot is not optional.

### The p-value

Every test in this notebook returns a p-value beside its statistic. It answers exactly one question:

> *If these two variables were genuinely unrelated, how often would I see a pattern this strong by
> luck alone?*

It does **not** say the relationship is strong, that it matters, or that it will hold next year.
Compare these two — both clear the usual 0.05 line.


In [ ]:
for col in ['temp in Celsius', 'humidity']:
    r, p_value = pearsonr(bike_df[col], bike_df['num_shared'])
    print(f"{col:>16}   r = {r:+.3f}   p = {p_value:.4f}   r-squared = {r**2:.3f}")

`humidity` explains about **1%** of the variation in daily rides. It clears 0.05 because there are
**731 days** — with enough rows, almost any correlation clears 0.05.

> **Report both, always. The coefficient says how strong; the p-value says how sure.**

And the reverse error, which is the one that persists: `p = 0.42` means *no evidence of a
relationship in this sample*. It is **not** proof that there is none.


---
### ✏️ Now You Try · 1

Boston Housing. Fill in the blanks — a `____` is something for you to replace.


**a)** Which columns in `boston_df` would rule out Pearson? Run `.skew()` and read the list.


In [ ]:
boston_df.____(numeric_only=True).sort_values()

**b)** `LSTAT` against `MEDV`. Check the skew of both, pick the test, run it, report the number.


In [ ]:
print(boston_df['LSTAT'].skew(), boston_df['MEDV'].skew())

r, p_value = ____(boston_df['LSTAT'], boston_df['MEDV'])
print(f"r = {r:.3f}, p = {p_value:.3g}")

**c)** Run the **other** test on the same pair too, then draw the scatterplot. They disagree by a lot — why?


In [ ]:
# your code here

**d)** In one sentence of your own words: what do the two numbers *together* tell you that
either one alone does not?


*Your answer:*


---
## 3 · A number, across groups

On Tuesday you drew this and said wet days are worse. You were reading the picture, not testing it.


In [ ]:
sns.boxplot(data=bike_df, x='weather', y='num_shared', order=['clear', 'cloudy', 'wet'])
plt.show()

bike_df.groupby('weather')['num_shared'].agg(['count', 'mean', 'median']).round(1)

clear 4877, cloudy 4036, wet 1803 — three different numbers. But there are only **21 wet days** out
of 731. How sure can you be that 1803 is a fact about rain, and not an accident of *which* 21 days
it happened to rain on?

**ANOVA** asks one question: *is the gap **between** the groups bigger than the wobble **within**
them?* That ratio is the F-statistic. Big F, the groups are genuinely apart. F near 1, they are as
far apart as noise alone would put them.

`f_oneway()` takes **one argument per group** — it cannot read a column of labels — so you have to
split the data yourself first. That splitting step is most of the work.


In [ ]:
clear  = bike_df[bike_df['weather'] == 'clear']['num_shared']
cloudy = bike_df[bike_df['weather'] == 'cloudy']['num_shared']
wet    = bike_df[bike_df['weather'] == 'wet']['num_shared']

print(len(clear), len(cloudy), len(wet))

f_stat, p_value = f_oneway(clear, cloudy, wet)
print(f"F = {f_stat:.3f}, p = {p_value:.3g}")

Read the filter line in English: *from `bike_df`, take the rows where weather is clear, then take the
`num_shared` column from those rows.*

F = 40.07. The gap between the groups is about **40 times** the wobble within them, and the p-value
is 3 × 10⁻¹⁷. Weather is real.

Note what `f_oneway` does **not** tell you: **which** group differs. It says at least one is
different. With three groups you look at the boxplot to see that `wet` is the outlier — a student
who reports "weather matters" without naming `wet` has not finished the job.

Now the same test on a question where there is nothing to find.


In [ ]:
groups = [bike_df[bike_df['weekday'] == d]['num_shared'] for d in range(7)]

f_stat, p_value = f_oneway(*groups)
print(f"F = {f_stat:.3f}, p = {p_value:.3f}")

sns.boxplot(data=bike_df, x='weekday', y='num_shared')
plt.show()

F = 0.78 — **below 1**, meaning the gaps between the days are smaller than the wobble inside them.
On Tuesday you guessed this from the boxplot; now it has a number.

And say it precisely: `p = 0.58` is **not** proof that weekday does not matter. It means 731 days
were not enough to find an effect, if there is one. *A non-significant result is still a result* —
"we checked and found nothing" is a legitimate, reportable finding.


---
### ✏️ Now You Try · 2

Back to Boston. `CHAS` is 1 if the tract touches the Charles River and 0 if it does not.


**a)** Split `MEDV` into the two groups.


In [ ]:
by_river  = boston_df[boston_df['CHAS'] == ____]['MEDV']
off_river = boston_df[boston_df['CHAS'] == ____]['MEDV']

**b)** How many tracts are in each group? Look **before** you test.


In [ ]:
# your code here

**c)** Run `f_oneway()` on the two groups. Report F and p.


In [ ]:
# your code here

**d)** Draw the boxplot. Then, in one sentence: would you tell a client that river frontage
raises home values?


In [ ]:
# your code here

*Your sentence:*


---
## 4 · Two categories

Neither column is a number, so there is nothing to average. You **count** instead — and the table of
counts is called a contingency table, or a cross-tabulation.


In [ ]:
table = pd.crosstab(index=bike_df['weather'], columns=bike_df['busyday'])
table

This is a pivot table with Count as the summary — the thing you built on Thursday, with a shorter
call. `pivot_table(index=..., columns=..., aggfunc='count')` would do the same job.

Read it: 387 ordinary clear days, 76 busy clear days, and — look at the bottom right — **zero** busy
wet days.

`margins=True` adds All rows and columns, which is useful for reading. **Never pass such a table to
the chi-square test**: the totals are not observations, and the test would count them as if they
were.


In [ ]:
pd.crosstab(index=bike_df['weather'], columns=bike_df['busyday'], margins=True)

### What would the table look like if the two were unrelated?

94 of the 731 days are busy — **12.9%** overall. If weather made no difference at all, you would
expect 12.9% of *each* weather group to be busy. That is only multiplication.


In [ ]:
share_busy = bike_df['busyday'].mean()
print(f"overall busy share: {share_busy:.3f}")

for w in ['clear', 'cloudy', 'wet']:
    n = (bike_df['weather'] == w).sum()
    print(f"{w:>7}: {n:3d} days  ->  expected busy {n * share_busy:5.1f}   observed {table.loc[w, 1]:3d}")

Expected 59.5 clear busy days and got 76. Expected 31.8 cloudy and got 18. Expected 2.7 wet and got
**none**.

**Chi-square is one number for how far the whole table drifted from what you expected.** Big gaps,
big chi-square, small p. And `chi2_contingency()` computes those expected counts for you — the ones
you just worked out by hand.


In [ ]:
chi2, p_value, dof, expected = chi2_contingency(observed=table)

print(f"chi2 = {chi2:.3f},  p = {p_value:.5f},  dof = {dof}")
print("\nexpected counts if weather and busyday were unrelated:")
print(pd.DataFrame(expected, index=table.index, columns=table.columns).round(1))

Four return values. `chi2` is the drift, `p_value` is whether it could be luck, `dof` is bookkeeping
— (rows − 1) × (columns − 1) — and `expected` is the array you just reproduced by hand. Most people
throw `expected` away. Do not: it is how you check the caveat below.

**In 731 days, not one busy day was ever a wet day.** On Tuesday you found that by noticing a
missing box on a chart. Thursday the wide pivot table showed it as a `NaN`. Today it has a p-value
of 0.0005. Three sessions, three tools, one finding.


In [ ]:
sns.countplot(data=bike_df, x='weather', hue='busyday',
              order=['clear', 'cloudy', 'wet'])
plt.ylabel('number of days')
plt.show()

**One caveat worth knowing.** Chi-square wants **every expected count to be at least 5**, and the
wet row expected 2.7. So this p-value is approximate. The finding is real and the direction is not
in doubt — but 21 days is thin evidence, and you would want more before betting money on it. This is
why you look at the `expected` array rather than discarding it.

And once more: rain does not care whether the city is busy. If anything the arrow runs the other way
— days are not busy *because* it is raining. Chi-square gives you neither direction.


---
### ✏️ Now You Try · 3

Still `bike_df`. Two categorical pairs; one of them is related and one is not.


**a)** Cross-tabulate `weekday` against `weather` and run the chi-square. What do you conclude?


In [ ]:
table_wd = pd.crosstab(index=bike_df['____'], columns=bike_df['____'])
display(table_wd)

chi2, p_value, dof, expected = chi2_contingency(table_wd)
print(f"chi2 = {chi2:.3f}, p = {p_value:.4f}, dof = {dof}")

**b)** Now do the same for `month` against `weather`.


In [ ]:
# your code here

**c)** One of the two came back significant. Which — and does that answer make sense to you?


*Your answer:*


**d)** In one sentence: **why** is that the one that came back significant?


In [ ]:
# a picture helps you answer (d)
# your code here

---
## 5 · Choosing a test

The whole session on one table. Start from your two variables, never from the test.

| What you have | Your question | The test | And always draw |
|---|---|---|---|
| two numbers, both symmetric | do they move together? | `pearsonr()` | `scatterplot()` |
| two numbers, one skewed | do they move together? | `spearmanr()` | `scatterplot()` |
| one number, one category | is it different by group? | `f_oneway()` | `boxplot(x=cat, y=num)` |
| two categories | are the two related? | `chi2_contingency()` | `countplot(x=, hue=)` |

**Every one of them returns two things: a statistic and a p-value.** The statistic says how strong.
The p-value says how sure.

- **Step 0** is `skew()` — it decides Pearson or Spearman before you correlate anything.
- **Step 1** is the picture.
- **Step 2** is the test. It never overrules the picture.

Three things none of these tests will do for you:

1. Tell you the relationship is **big enough to care about**. That is the coefficient's job, and
   yours.
2. Tell you **which** group differs, when ANOVA fires on three or more groups. Read the boxplot.
3. Establish **cause**. All three rule out luck. That is a much smaller claim than it sounds, and
   it is the one everybody in the room will assume you have made.

---

Week 3 homework is on Canvas, due Sunday 11:59 pm. Thursday is *what machine learning is* — bring
nothing; there is no notebook until the last ten minutes.
